In [ ]:
import numpy as np
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

import napari
from skimage import io

from morphotrack import losses, networks, analysis, utils
from geomloss import SamplesLoss

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

import trimesh
from tqdm import tqdm

import open3d as o3d
import zarr

In [ ]:
fix_path = '/mnt/ampa_data01/tmurakami/220806_visual_02_R01/R02_R01/R02ch488_to_R01.zarr'
resolution = 4
fix_chan = 1
fix_img = zarr.open(fix_path)[resolution][:]

sma_path = '/mnt/ampa_data01/tmurakami/220806_visual_02_R01/ch561.zarr'
sma_img = zarr.open(sma_path)[resolution][:]

scale = np.asarray([3.0,0.65,0.65])
downsample_factor = np.asarray([16,16,16])
unit_size = downsample_factor * scale

In [ ]:
# Load coordinates and vectors
coords = np.load('/home/tmurakami/src/flow_analysis/human_analysis/02_output/visual_02_R01/positions_sma_dense.npy')
local_vectors = np.load('/home/tmurakami/src/flow_analysis/human_analysis/02_output/visual_02_R01/vectors_sma_dense.npy')
local_vectors = local_vectors / np.linalg.norm(local_vectors, axis=1, keepdims=True)

fix_mesh = trimesh.load("/home/tmurakami/src/flow_analysis/human_analysis/01_output/220806_visual_02_R01_pia_refined.ply")
mov_mesh = trimesh.load("/home/tmurakami/src/flow_analysis/human_analysis/01_output/220806_visual_02_R01_wm_refined.ply")
fix_vertices = fix_mesh.vertices
mov_vertices = mov_mesh.vertices
fix_face = fix_mesh.faces
mov_face = mov_mesh.faces

# Get test points on the mov surface
nop = 3000
# Convert Trimesh to Open3D
o3d_mesh = o3d.geometry.TriangleMesh()
o3d_mesh.vertices = o3d.utility.Vector3dVector(mov_mesh.vertices)
o3d_mesh.triangles = o3d.utility.Vector3iVector(mov_mesh.faces)
mov_test = o3d_mesh.sample_points_uniformly(number_of_points=nop)
mov_test = np.asarray(mov_test.points)

### Parameter settings
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Predict unit velocity field

In [ ]:
lr=1e-3

mixed_orientation = False
flipvector = False

v_field = networks.SimpleMLP2(hidden_sizes=[256, 128, 64], use_norm=True,
                              use_residual=True, activation_func='SiLU').to(device)

optimizer = torch.optim.AdamW(v_field.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5000, min_lr=1e-5)
loss_log = {'align': [], 'consistency': [], 'smooth': [], 'divergence': [], 'curl': [], 'laplacian': [], 'dirichlet': [], 'curvature': [], 'torsion': []}

vec_pos = torch.from_numpy(coords).to(device, dtype=torch.float32)

### bbox bounds for normalization
all_verts_np = np.vstack([mov_mesh.vertices, fix_mesh.vertices])
pos_min = torch.from_numpy(all_verts_np.min(axis=0)).to(device, dtype=torch.float32)
pos_max = torch.from_numpy(all_verts_np.max(axis=0)).to(device, dtype=torch.float32)

vec_pos_norm = (vec_pos - pos_min) / (pos_max - pos_min)  # [0, 1]

dGdt_unit = torch.from_numpy(local_vectors).to(device, dtype=torch.float32)

fix_original = torch.from_numpy(np.asarray(fix_vertices)).to(device, dtype=torch.float32)
mov_test_original = torch.from_numpy(np.asarray(mov_test)).to(device, dtype=torch.float32)

if flipvector:
    local_vectors = -local_vectors

checkpoint_path = 'v_field_checkpoint_temp.pth'
best_loss = float('inf')  # Initialize best loss

In [ ]:
# Generate the random points between the two surfaces
from scipy.spatial import cKDTree

def sample_points_between_surfaces(mesh_a, mesh_b, n_candidates, use_bbox=False):
    tree_a = cKDTree(mesh_a.vertices)
    tree_b = cKDTree(mesh_b.vertices)

    all_verts = np.vstack([mesh_a.vertices, mesh_b.vertices])
    bbox_min = all_verts.min(axis=0)
    bbox_max = all_verts.max(axis=0)

    candidates = np.random.uniform(bbox_min, bbox_max, size=(n_candidates, 3))

    if use_bbox:
        return candidates

    _, idx_a = tree_a.query(candidates)
    _, idx_b = tree_b.query(candidates)

    closest_a = tree_a.data[idx_a]
    closest_b = tree_b.data[idx_b]

    vec_ab = closest_b - closest_a
    vec_ac = candidates - closest_a

    dot_ab = np.einsum('ij,ij->i', vec_ab, vec_ab)
    dot_ac = np.einsum('ij,ij->i', vec_ab, vec_ac)

    t = np.where(dot_ab > 1e-12, dot_ac / np.maximum(dot_ab, 1e-12), -1.0)
    mask = (t > 0.0) & (t < 1.0)

    return candidates[mask]

r_points_original_np = sample_points_between_surfaces(fix_mesh, mov_mesh, n_candidates=500000, use_bbox=True)  
r_points_original = torch.from_numpy(r_points_original_np).to(device, dtype=torch.float32)
r_points_norm = (r_points_original - pos_min) / (pos_max - pos_min)
# r_points_norm = (r_points_original - pos_min) / s_max


In [ ]:
viewer = napari.Viewer()
viewer.add_points(fix_vertices,size=unit_size[0]*40/25.,face_color='lime',blending='additive',out_of_slice_display=True)
viewer.add_points(mov_test,size=unit_size[0]*40/25.,face_color='magenta',blending='additive',out_of_slice_display=True)
viewer.add_vectors(np.stack([coords, local_vectors], axis=1), length=unit_size[0]*5., edge_width=unit_size[0]*2., edge_color='red', name='3D Vectors',out_of_slice_display=True)
viewer.add_image(fix_img, name='Reference Image', blending='additive', scale=unit_size)
viewer.add_image(sma_img, name='SMA Image', blending='additive', scale=unit_size)
viewer.add_points(r_points_original_np,size=unit_size[0]*40/25.,face_color='white',blending='additive',out_of_slice_display=True)
# viewer.add_surface((mov_vertices, mov_face), name='WM Mesh', blending='translucent',shading='smooth')
# viewer.add_surface((fix_vertices, fix_face), name='Pia Mesh', blending='additive',shading='smooth')

In [ ]:
def approx_const_curvature_loss(v_field, points, epsilon=1e-3):
    v0 = F.normalize(v_field(points), p=2, dim=1)
    x1 = points + epsilon * v0
    v1 = F.normalize(v_field(x1), p=2, dim=1)
    x2 = x1 + epsilon * v1
    v2 = F.normalize(v_field(x2), p=2, dim=1)
    return ((v2 - 2 * v1 + v0) ** 2).sum(dim=1).mean()  # ≈ ε⁴ · ‖d²v̂/ds²‖²

def approx_zero_torsion_loss(v_field, points, epsilon=1e-3):
    """Penalize (T · (T' × T''))² ≈ κ⁴ τ² along streamlines.
    Drives streamlines to be locally planar (zero torsion)
    or straight (zero curvature). Weighted by curvature⁴ — natural
    handling of straight regions (no division)."""
    v0 = F.normalize(v_field(points), p=2, dim=1)
    x1 = points + epsilon * v0
    v1 = F.normalize(v_field(x1), p=2, dim=1)
    x2 = x1 + epsilon * v1
    v2 = F.normalize(v_field(x2), p=2, dim=1)

    T        = v1
    T_prime  = (v2 - v0) / (2 * epsilon)
    T_dprime = (v2 - 2 * v1 + v0) / (epsilon ** 2)

    triple = (T * torch.cross(T_prime, T_dprime, dim=1)).sum(dim=1)   # scalar per pt
    return triple.pow(2).mean()

def dirichlet_energy(net, X):
    """Monte-Carlo estimate of ∫|∇φ|² over interior samples X. (Deep Ritz.)"""
    Xg  = X.detach().requires_grad_(True)
    phi = net(Xg).squeeze(-1)
    grad_phi = torch.autograd.grad(
        phi.sum(), Xg, create_graph=True, retain_graph=True
    )[0]                                                   # (N, 3)
    return grad_phi.pow(2).sum(dim=-1).mean()


## Parameter settings for training
n_iters=5000000
lambda_align = 1.0 
lambda_smooth = 0
lambda_divergence = 0 # 1e-4
lambda_curl = 0
lambda_lipschitz = 0
lambda_dirichlet = 1e-2 # Make it look Laplacian harmonic
lambda_curvature = 0
lambda_torsion = 1e-6 # Tortion free regularization
sampling_size = 5000

In [ ]:
# Pre-normalize mov/fix once.
mov_test_norm = (mov_test_original - pos_min) / (pos_max - pos_min)
fix_norm      = (fix_original     - pos_min) / (pos_max - pos_min)

# -----------------------
# Train
# -----------------------

v_field.train()
for it in range(n_iters):
    optimizer.zero_grad()

    # Compute alignment loss
    v_pred = v_field(vec_pos_norm)
    v_pred_norm = v_pred / (v_pred.norm(dim=1, keepdim=True))
    cos_sim = torch.sum(v_pred_norm * dGdt_unit, dim=1)
    loss_align = (1 - cos_sim).mean()

    # Get position of the trajectory for smoothness loss
    idx = torch.randint(len(r_points_norm), (sampling_size,))
    r_points = r_points_norm[idx]
    r_points = r_points.clone().requires_grad_(True)

    # For Laplacian
    velocities = v_field(r_points)
    mag_v   = velocities.norm(dim=1).clamp(min=1e-6)
    N, D = velocities.shape

    jacobian = []

    for i in range(D):  # vx, vy, vz
        grad_i = torch.autograd.grad(
            outputs=velocities[:, i],
            inputs=r_points,
            grad_outputs=torch.ones_like(velocities[:, i]),
            create_graph=True,
            retain_graph=True,
            only_inputs=True,
        )[0]  # Shape: [N, 3]
        jacobian.append(grad_i)

    # Stack into Jacobian tensor [N, 3, 3]
    jacobian = torch.stack(jacobian, dim=1)

    # Smoothness Loss
    loss_smooth = ((jacobian ** 2).sum(dim=(1,2)) / mag_v.pow(2)).mean() # (jacobian ** 2).sum(dim=(1, 2)).mean()

    # Lipschitz loss
    singular_values = torch.linalg.svdvals(jacobian)
    spectral_norms = singular_values[:, 0]
    loss_lipschitz = (spectral_norms / mag_v).mean() # spectral_norms.mean()

    # Divergence Loss
    divergence = torch.stack([
        jacobian[:, 0, 0],  # dvx/dx
        jacobian[:, 1, 1],  # dvy/dy
        jacobian[:, 2, 2],  # dvz/dz
    ], dim=1).sum(dim=1)
    loss_divergence = ((divergence / mag_v) ** 2).mean() # (divergence ** 2).mean()

    # Curl Loss
    curl_x = jacobian[:, 2, 1] - jacobian[:, 1, 2]
    curl_y = jacobian[:, 0, 2] - jacobian[:, 2, 0]
    curl_z = jacobian[:, 1, 0] - jacobian[:, 0, 1]
    curl = torch.stack([curl_x, curl_y, curl_z], dim=1)
    loss_curl = ((curl ** 2).sum(dim=1) / mag_v.pow(2)).mean() # (curl ** 2).sum(dim=1).mean()

    loss_dirichlet = dirichlet_energy(v_field, r_points)
    loss_curvature = approx_const_curvature_loss(v_field, r_points, epsilon=1e-3)
    loss_torsion = approx_zero_torsion_loss(v_field, r_points, epsilon=1e-3)


    # Compute total loss
    loss = (lambda_align * loss_align
            + lambda_smooth * loss_smooth
            + lambda_divergence * loss_divergence
            + lambda_curl * loss_curl
            + lambda_lipschitz * loss_lipschitz
            + lambda_dirichlet * loss_dirichlet
            + lambda_curvature * loss_curvature
            + lambda_torsion * loss_torsion
            )


    loss.backward()
    torch.nn.utils.clip_grad_norm_(v_field.parameters(), max_norm=1.0)
    optimizer.step()
    # scheduler.step()
    scheduler.step(loss_align.item())

    # Logging
    loss_log['align'].append(loss_align.item())
    loss_log['divergence'].append(loss_divergence.item())
    loss_log['curl'].append(loss_curl.item())
    loss_log['smooth'].append(loss_smooth.item())
    loss_log['dirichlet'].append(loss_dirichlet.item())
    loss_log['curvature'].append(loss_curvature.item())
    loss_log['torsion'].append(loss_torsion.item())

    # Update lambda schedulers
    current_losses = {
        'align': loss_align.item(),
        'smooth': loss_smooth.item(),
        'divergence': loss_divergence.item(),
        'curl': loss_curl.item(),
        'dirichlet': loss_dirichlet.item(),
        'curvature': loss_curvature.item(),
        'torsion': loss_torsion.item(),
    }

    if it % 50 == 0:
        msg = (f"[{it}] Total: {loss.item():.4f}, "
               f"Align: {loss_align.item():.4f}, "
               f"Smooth(x{lambda_smooth:.4f}): {loss_smooth.item():.6f}, "
               f"Div(x{lambda_divergence:.4f}): {loss_divergence.item():.6f}, "
               f"Curl(x{lambda_curl:.4f}): {loss_curl.item():.6f}, "
               f"Lipschitz(x{lambda_lipschitz:.4f}): {loss_lipschitz.item():.6f}, "
               f"Dirichlet(x{lambda_dirichlet}): {loss_dirichlet.item()}, "
               f"Curvature(x{lambda_curvature}): {loss_curvature.item()}, "
               f"Torsion(x{lambda_torsion}): {loss_torsion.item()}, "
               )

        print(msg)


    # Checkpoint saving
    if loss.item() < best_loss:
        best_loss = loss.item()
        torch.save({
            'iteration': it,
            'model_state_dict': v_field.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item(),
        }, checkpoint_path)

In [ ]:
iters = range(len(loss_log['align']))

fig, axes = plt.subplots(ncols=3, figsize=(14, 5))  # 2 plots side-by-side

# Plot Match Loss on the left
axes[0].plot(iters, loss_log['align'], label='Align Loss')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_yscale('log')
axes[0].set_title('Loss Over Iterations')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(iters, loss_log['smooth'], label='Smooth Loss')
axes[1].plot(iters, loss_log['divergence'], label='Divergence Loss')
axes[1].plot(iters, loss_log['curl'], label='Curl Loss')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss')
axes[1].set_yscale('log')
axes[1].set_title('Loss Over Iterations')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(iters, loss_log['curvature'], label='Curvature Loss')
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Loss')
axes[2].set_yscale('log')
axes[2].set_title('Loss Over Iterations')
axes[2].legend()
axes[2].grid(True)

# # Plot each lambda scheduler
# for i, ls in enumerate(lambda_schedulers):
#     ls.plot(ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
# Load v_field checkpoint
checkpoint_path = './v_field_checkpoint_temp.pth'# 'v_field_checkpoint_temp_1e5.pth' # 
v_field_ckpt = torch.load(checkpoint_path, map_location='cuda')
v_field.load_state_dict(v_field_ckpt['model_state_dict'])
v_field.eval()  # Set to evaluation mode

In [ ]:
# -----------------------
# Visualization
# -----------------------
class NormalizedField:
    def __init__(self, model, pos_min, pos_max):
        self.model = model
        self.pos_min = pos_min
        self.pos_max = pos_max
    def __call__(self, x):
        x_norm = (x - self.pos_min) / (self.pos_max - self.pos_min)
        return F.normalize(self.model(x_norm), p=2, dim=1)
    def eval(self):
        self.model.eval()
        return self
    
v_field_norm = NormalizedField(v_field, pos_min, pos_max)

v_field_norm.eval()
thickness = 4000 # in microns, this will be final time
steps = 50
dt = thickness/steps
max_lines= 500

with torch.no_grad():
    mov_traj = analysis.integrate_rk4(mov_test_original, v_field_norm, steps=steps, dt=dt)

    X_all = mov_traj.detach().cpu().numpy()  # (T+1, N, 3)
    _, N, _ = X_all.shape

    # Determine which trajectories to draw as lines
    idx = utils.random_idx_with_max(N, max_lines)
    target=fix_original.detach().cpu().numpy()
    size = unit_size[0]*30/25
    edge_width = 4
    mov_solid_color = np.tile([0.8, 0.0, 0.8], (len(mov_vertices), 1))
    fix_solid_color = np.tile([0.0, 0.8, 0.0], (len(fix_vertices), 1))

    viewer = napari.Viewer(ndisplay=3)
    
    shapes = [X_all[:, i, :] for i in range(X_all.shape[1])] if idx is None else [X_all[:, i, :] for i in idx]

    # Sampled initial points (magenta)
    viewer.add_shapes(shapes, shape_type='path', edge_color='yellow', edge_width=edge_width, name='Trajectories', visible=False)
    viewer.add_surface((mov_vertices, mov_face), name='WM Mesh', vertex_colors=mov_solid_color, blending='translucent',shading='smooth')
    # viewer.add_points(X_all[0], name='X (t=0)', size=size, face_color='magenta', blending='additive')
    viewer.add_points(X_all[-1], name='X (t=1) (all)', size=size, face_color='red', blending='additive', visible=False)
    # viewer.add_points(target, name='Target', size=size, face_color='lime', blending='additive')
    viewer.add_surface((fix_vertices, fix_face), name='Pia Mesh', vertex_colors=fix_solid_color, blending='additive',shading='smooth')

    # Compute alignment loss and visualize orientation arrows
    v_pred = v_field_norm(vec_pos)
    alignment_sim = losses.alignment_loss(v_pred, dGdt_unit, norm_vector=True, norm_ref=True, mixed_orientation=mixed_orientation, return_vector=True).detach().cpu().numpy()
    cmap = plt.get_cmap('cool')  # You can use 'plasma', 'magma', etc.

    # Map scalar values to RGBA colors using the colormap
    alignment_sim_norm = alignment_sim / np.percentile(alignment_sim, 50)
    alignment_sim_norm[alignment_sim_norm>1] = 1
    colors = cmap(alignment_sim_norm)  # Output shape: (10, 4), with RGBA in [0, 1]
    viewer.add_vectors(np.stack([coords, local_vectors], axis=1), edge_color=colors, name='orientation_arrows', length=unit_size[0]*100/25., edge_width=unit_size[0]*10/25., visible=False)

    shapes = []
    for i in tqdm(range(mov_traj.shape[1])):
        trajectory = mov_traj[:,i,:].detach().cpu().numpy()
        hit, intersection_time = analysis.calculate_intersection(trajectory, fix_mesh)
        if intersection_time is not np.nan:
            if int(intersection_time) > 1:
                shapes.append(trajectory[:int(intersection_time)])

    idx = utils.random_idx_with_max(len(shapes), max_lines)
    shapes = [shapes[i] for i in idx]

    viewer.add_shapes(shapes, shape_type='path', edge_color='yellow', edge_width=edge_width, name='Trajectories')